In [31]:
import os
import json
import calendar
from pathlib import Path

import cdsapi

# ==========================================
# 0. 下载任务配置（可按需调整）
# ==========================================
YEARS = ["2018"]
MONTHS = [f"{m:02d}" for m in range(7, 13)]

# 覆盖 Station 00-09 的经纬度大外框 (北, 西, 南, 东)
AREA = [40.0, 113.0, 36.0, 118.0]

DOWNLOAD_DIR = Path("./era5_raw_data")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("配置加载完成")
print("YEARS:", YEARS)
print("MONTHS:", MONTHS[:3], "...", MONTHS[-3:])
print("AREA:", AREA)
print("DOWNLOAD_DIR:", DOWNLOAD_DIR.resolve())

配置加载完成
YEARS: ['2018']
MONTHS: ['07', '08', '09'] ... ['10', '11', '12']
AREA: [40.0, 113.0, 36.0, 118.0]
DOWNLOAD_DIR: /Users/dazhaxie/Desktop/Deye/PVOD_Forecast/feature_engineering/era5_raw_data


In [32]:
# ==========================================
# 1. 请求构建与参数校验
# ==========================================

def build_days(year: str, month: str) -> list[str]:
    n_days = calendar.monthrange(int(year), int(month))[1]
    return [f"{d:02d}" for d in range(1, n_days + 1)]

def build_times() -> list[str]:
    # 不降低时间分辨率：保持逐小时
    return [f"{h:02d}:00" for h in range(24)]

def chunk_list(items: list[str], chunk_size: int) -> list[list[str]]:
    if chunk_size <= 0:
        raise ValueError("chunk_size 必须 > 0")
    return [items[i:i + chunk_size] for i in range(0, len(items), chunk_size)]

def build_request_2d(year: str, month: str) -> dict:
    return {
        "product_type": "reanalysis",
        "variable": [
            "surface_solar_radiation_downwards",
            "total_sky_direct_solar_radiation_at_surface",
            "2m_temperature",
            "2m_dewpoint_temperature",
            "total_cloud_cover",
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "surface_pressure",
            "total_precipitation",
            "low_cloud_cover",
            "medium_cloud_cover",
            "high_cloud_cover",
            "total_column_water_vapour",
            "boundary_layer_height"
        ],
        "year": year,
        "month": month,
        "day": build_days(year, month),
        "time": build_times(),
        "area": AREA,
        "format": "netcdf",
    }

def build_request_3d(
    year: str,
    month: str,
    variables: list[str],
    pressure_levels: list[str],
    days: list[str],
) -> dict:
    return {
        "product_type": "reanalysis",
        "variable": variables,
        "pressure_level": pressure_levels,
        "year": year,
        "month": month,
        "day": days,
        "time": build_times(),
        "area": AREA,
        "format": "netcdf",
    }

ALL_3D_VARIABLES = [
    "relative_humidity",
    "temperature",
    "u_component_of_wind",
    "v_component_of_wind",
    "vertical_velocity",
    "geopotential",
    "specific_cloud_liquid_water_content",
    "specific_cloud_ice_water_content",
]
ALL_3D_PRESSURE_LEVELS = ["500", "700", "850", "925", "1000"]

# 快速校验
_sample_2d = build_request_2d(YEARS[0], MONTHS[0])
_sample_3d = build_request_3d(
    YEARS[0],
    MONTHS[0],
    variables=ALL_3D_VARIABLES[:3],
    pressure_levels=ALL_3D_PRESSURE_LEVELS[:2],
    days=build_days(YEARS[0], MONTHS[0]),
)
print("请求构建校验通过")
print("2D day count:", len(_sample_2d["day"]), "| time count:", len(_sample_2d["time"]))
print("3D(day/time/vars/pl):", len(_sample_3d["day"]), len(_sample_3d["time"]), len(_sample_3d["variable"]), len(_sample_3d["pressure_level"]))

请求构建校验通过
2D day count: 31 | time count: 24
3D(day/time/vars/pl): 31 24 3 2


In [33]:
# ==========================================
# 2. 下载执行函数（支持 dry-run + 3D 自动分块 + 先提交后下载）
# ==========================================

def download_era5_data(
    years: list[str],
    months: list[str],
    dry_run: bool = True,
    limit_jobs: int | None = None,
    skip_existing: bool = True,
    download_dir: Path = DOWNLOAD_DIR,
    split_3d: bool = True,
    split_strategy: str = "var_group",
    var_chunk_size: int = 3,
    pl_chunk_size: int = 2,
    day_chunk_size: int | None = None,
    submit_only: bool = False,
    manifest_path: Path | None = None,
) -> int:
    client = None if dry_run else cdsapi.Client(wait_until_complete=not submit_only)
    submitted = 0
    manifest_records = []

    for year in years:
        for month in months:
            file_2d = download_dir / f"era5_2d_{year}_{month}.nc"
            if (not skip_existing) or (not file_2d.exists()):
                req_2d = build_request_2d(year, month)
                if dry_run:
                    print(f"[DRY-RUN] 2D {year}-{month} -> {file_2d.name} (days={len(req_2d['day'])})")
                else:
                    print(f"提交下载任务: 2D {year}-{month}")
                    result_2d = client.retrieve("reanalysis-era5-single-levels", req_2d, None if submit_only else str(file_2d))
                    if submit_only:
                        manifest_records.append({
                            "target": str(file_2d),
                            "collection": "reanalysis-era5-single-levels",
                            "request": req_2d,
                            "request_id": result_2d.request_id,
                        })
                submitted += 1
                if limit_jobs is not None and submitted >= limit_jobs:
                    print(f"达到 limit_jobs={limit_jobs}，提前结束。")
                    break

            month_days = build_days(year, month)
            day_chunks = [month_days] if day_chunk_size is None else chunk_list(month_days, day_chunk_size)

            if not split_3d:
                file_3d = download_dir / f"era5_3d_{year}_{month}.nc"
                if (not skip_existing) or (not file_3d.exists()):
                    req_3d = build_request_3d(
                        year, month, ALL_3D_VARIABLES, ALL_3D_PRESSURE_LEVELS, month_days
                    )
                    if dry_run:
                        print(f"[DRY-RUN] 3D {year}-{month} -> {file_3d.name}")
                    else:
                        print(f"提交下载任务: 3D {year}-{month}")
                        result_3d = client.retrieve("reanalysis-era5-pressure-levels", req_3d, None if submit_only else str(file_3d))
                        if submit_only:
                            manifest_records.append({
                                "target": str(file_3d),
                                "collection": "reanalysis-era5-pressure-levels",
                                "request": req_3d,
                                "request_id": result_3d.request_id,
                            })
                    submitted += 1
                    if limit_jobs is not None and submitted >= limit_jobs:
                        print(f"达到 limit_jobs={limit_jobs}，提前结束。")
                        break
                continue

            if split_strategy == "var_group":
                var_chunks = chunk_list(ALL_3D_VARIABLES, var_chunk_size)
                for v_idx, vars_part in enumerate(var_chunks, start=1):
                    for d_idx, days_part in enumerate(day_chunks, start=1):
                        day_tag = "full" if day_chunk_size is None else f"d{d_idx:02d}"
                        file_3d = download_dir / f"era5_3d_{year}_{month}_vg{v_idx}_{day_tag}.nc"
                        if skip_existing and file_3d.exists():
                            continue
                        req_3d = build_request_3d(
                            year, month, vars_part, ALL_3D_PRESSURE_LEVELS, days_part
                        )
                        if dry_run:
                            print(
                                f"[DRY-RUN] 3D {year}-{month} -> {file_3d.name} "
                                f"(vars={len(vars_part)}, pl={len(ALL_3D_PRESSURE_LEVELS)}, days={len(days_part)})"
                            )
                        else:
                            print(f"提交下载任务: 3D {year}-{month} vg{v_idx} {day_tag}")
                            result_3d = client.retrieve("reanalysis-era5-pressure-levels", req_3d, None if submit_only else str(file_3d))
                            if submit_only:
                                manifest_records.append({
                                    "target": str(file_3d),
                                    "collection": "reanalysis-era5-pressure-levels",
                                    "request": req_3d,
                                    "request_id": result_3d.request_id,
                                })
                        submitted += 1
                        if limit_jobs is not None and submitted >= limit_jobs:
                            print(f"达到 limit_jobs={limit_jobs}，提前结束。")
                            break

            elif split_strategy == "pressure_level":
                pl_chunks = chunk_list(ALL_3D_PRESSURE_LEVELS, pl_chunk_size)
                for p_idx, pl_part in enumerate(pl_chunks, start=1):
                    for d_idx, days_part in enumerate(day_chunks, start=1):
                        day_tag = "full" if day_chunk_size is None else f"d{d_idx:02d}"
                        file_3d = download_dir / f"era5_3d_{year}_{month}_pl{p_idx}_{day_tag}.nc"
                        if skip_existing and file_3d.exists():
                            continue
                        req_3d = build_request_3d(
                            year, month, ALL_3D_VARIABLES, pl_part, days_part
                        )
                        if dry_run:
                            print(
                                f"[DRY-RUN] 3D {year}-{month} -> {file_3d.name} "
                                f"(vars={len(ALL_3D_VARIABLES)}, pl={len(pl_part)}, days={len(days_part)})"
                            )
                        else:
                            print(f"提交下载任务: 3D {year}-{month} pl{p_idx} {day_tag}")
                            result_3d = client.retrieve("reanalysis-era5-pressure-levels", req_3d, None if submit_only else str(file_3d))
                            if submit_only:
                                manifest_records.append({
                                    "target": str(file_3d),
                                    "collection": "reanalysis-era5-pressure-levels",
                                    "request": req_3d,
                                    "request_id": result_3d.request_id,
                                })
                        submitted += 1
                        if limit_jobs is not None and submitted >= limit_jobs:
                            print(f"达到 limit_jobs={limit_jobs}，提前结束。")
                            break

            else:
                raise ValueError("split_strategy 仅支持 'var_group' 或 'pressure_level'")

        if limit_jobs is not None and submitted >= limit_jobs:
            break

    if submit_only and manifest_path is not None and manifest_records:
        with open(manifest_path, "w", encoding="utf-8") as f:
            json.dump(manifest_records, f, ensure_ascii=False, indent=2)
        print(f"已保存提交清单: {manifest_path}")

    return submitted


def download_from_manifest(manifest_path: Path) -> int:
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest_records = json.load(f)

    client = cdsapi.Client()
    results = [record["request_id"] for record in manifest_records]
    targets = [record["target"] for record in manifest_records]

    print(f"准备下载已提交任务: {len(results)} 个")
    for request_id, target in zip(results, targets):
        print(f"下载 {request_id} -> {target}")
        client.download_results(request_id, target)
    return len(results)

In [34]:
# ==========================================
# 3. 两阶段入口
#    1) 夜间 submit_only：只提交任务并保存清单
#    2) 早上 download：按清单一键下载已完成结果
# ==========================================

YEARS_TO_RUN = YEARS
MONTHS_TO_RUN = MONTHS
MANIFEST_PATH = DOWNLOAD_DIR / "era5_submission_manifest.json"

# 3D 自动分块策略（不缩小区域，不降低时间分辨率）
SPLIT_3D = True
SPLIT_STRATEGY = "pressure_level"   # "var_group" 或 "pressure_level"
VAR_CHUNK_SIZE = 2
PL_CHUNK_SIZE = 1
DAY_CHUNK_SIZE = None

# ---------- 阶段 A：夜间提交任务 ----------
SUBMIT_ONLY = True
if SUBMIT_ONLY:
    submitted_jobs = download_era5_data(
        years=YEARS_TO_RUN,
        months=MONTHS_TO_RUN,
        dry_run=False,
        limit_jobs=None,
        skip_existing=True,
        download_dir=DOWNLOAD_DIR,
        split_3d=SPLIT_3D,
        split_strategy=SPLIT_STRATEGY,
        var_chunk_size=VAR_CHUNK_SIZE,
        pl_chunk_size=PL_CHUNK_SIZE,
        day_chunk_size=DAY_CHUNK_SIZE,
        submit_only=True,
        manifest_path=MANIFEST_PATH,
    )
    print(f"已提交任务数={submitted_jobs}")
    print(f"提交清单已写入: {MANIFEST_PATH}")

# ---------- 阶段 B：早上下载结果 ----------
DOWNLOAD_ONLY = False
if DOWNLOAD_ONLY:
    downloaded_jobs = download_from_manifest(MANIFEST_PATH)
    print(f"已下载任务数={downloaded_jobs}")

print("入口已准备好：需要提交时运行 SUBMIT_ONLY 段，需要下载时运行 DOWNLOAD_ONLY 段。")
print(f"3D 分块: split={SPLIT_3D}, strategy={SPLIT_STRATEGY}, day_chunk={DAY_CHUNK_SIZE}")

提交下载任务: 2D 2018-07


2026-04-22 00:51:44,194 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-04-22 00:51:44,196 INFO Request ID is 712a2fb3-ed5b-407c-b20f-10f710f9b439


提交下载任务: 3D 2018-07 pl1 full


2026-04-22 00:51:44,695 INFO Request ID is 9e97d2ae-8e0e-4234-b366-579cde50d260


提交下载任务: 3D 2018-07 pl2 full


...

提交下载任务: 3D 2018-12 pl4 full


2026-04-22 00:52:03,835 INFO Request ID is 9127f939-e29d-4353-ac4f-cb8ad413cd51


提交下载任务: 3D 2018-12 pl5 full


2026-04-22 00:52:04,327 INFO Request ID is f039b143-56b4-4ddd-ba38-fb80abaf1e90


已保存提交清单: era5_raw_data/era5_submission_manifest.json
已提交任务数=36
提交清单已写入: era5_raw_data/era5_submission_manifest.json
入口已准备好：需要提交时运行 SUBMIT_ONLY 段，需要下载时运行 DOWNLOAD_ONLY 段。
3D 分块: split=True, strategy=pressure_level, day_chunk=None
